## 구글 드라이브 마운트 & 파일 경로 입력

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
seismic_path = "/content/drive/MyDrive/TriAI/그로쓰/Data/processed/multimodal/(2)지진파형 정렬/tohoku_seismic_360_180_ver2.npz"
gnss_path = "/content/drive/MyDrive/TriAI/그로쓰/Data/processed/gnss/tohoku/360_180/1hz/tohoku_gnss_pgv_dataset_25km_seq.npz"
pair_path = "/content/drive/MyDrive/TriAI/그로쓰/Data/csv/station_pairs/tohoku/tohoku_station_pairs_ver_25km.csv"

## 데이터 확인

In [ ]:
import numpy as np
import pandas as pd

tohoku_seismic = np.load(seismic_path)
tohoku_gnss = np.load(gnss_path)
tohoku_pair = pd.read_csv(pair_path)

In [ ]:
# 지진파형
for key in tohoku_seismic.files:
    print(f"Key: {key}, Shape: {tohoku_seismic[key].shape}, Dtype: {tohoku_seismic[key].dtype}")

Key: data, Shape: (131, 14, 36000, 3), Dtype: float64
Key: station_names, Shape: (131,), Dtype: <U6
Key: start_times, Shape: (131, 14), Dtype: <U27


In [ ]:
# GNSS
for key in tohoku_gnss.files:
    print(f"Key: {key}, Shape: {tohoku_gnss[key].shape}, Dtype: {tohoku_gnss[key].dtype}")

Key: X, Shape: (172, 14, 360, 3), Dtype: float32
Key: y, Shape: (172,), Dtype: float32
Key: gnss_station, Shape: (172,), Dtype: <U8
Key: seismic_station, Shape: (172,), Dtype: <U6
Key: start_sec, Shape: (172, 14), Dtype: float32
Key: end_sec, Shape: (172, 14), Dtype: float32
Key: fs, Shape: (172,), Dtype: float32
Key: gnss_lat, Shape: (172,), Dtype: float32
Key: gnss_lon, Shape: (172,), Dtype: float32


In [ ]:
# 지진파형
seismic_data = tohoku_seismic['data']
seismic_station = tohoku_seismic['station_names']
seismic_time = tohoku_seismic['start_times']

print(seismic_data.shape)

# GNSS
gnss_data = tohoku_gnss['X']
gnss_station = tohoku_gnss['gnss_station']
gnss_time = tohoku_gnss['start_sec']

print(gnss_data.shape)

(131, 14, 36000, 3)
(172, 14, 360, 3)


In [ ]:
print(seismic_time)
print('=============')
print(gnss_time)

[['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
  '2011-03-11T14:51:00.000000Z' ... '2011-03-11T15:18:00.000000Z'
  '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
 ['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
  '2011-03-11T14:51:00.000000Z' ... '2011-03-11T15:18:00.000000Z'
  '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
 ['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
  '2011-03-11T14:51:00.000000Z' ... '2011-03-11T15:18:00.000000Z'
  '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
 ...
 ['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
  '2011-03-11T14:51:00.000000Z' ... '2011-03-11T15:18:00.000000Z'
  '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
 ['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
  '2011-03-11T14:51:00.000000Z' ... '2011-03-11T15:18:00.000000Z'
  '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
 ['2011-03-11T14:45:00.000000Z' '2011-03

In [ ]:
tohoku_pair['seismic_station'].value_counts()

,count
seismic_station,
N.YWTH,4
N.TAJH,4
N.IWNH,4
N.KMOH,4
N.KUCH,3
...,...
N.NSEH,1
N.ATKH,1
N.MRUH,1


지진파형 관측소 기준 중복횟수 내림차순으로 pair 정렬

In [ ]:
counts = tohoku_pair['seismic_station'].value_counts()

sorted = tohoku_pair.assign(
    station_count = tohoku_pair['seismic_station'].map(counts)
).sort_values(by='station_count', ascending=False).drop(columns='station_count')

In [ ]:
print(counts)

seismic_station
N.YWTH    4
N.TAJH    4
N.IWNH    4
N.KMOH    4
N.KUCH    3
         ..
N.NSEH    1
N.ATKH    1
N.MRUH    1
N.KYWH    1
N.NYAH    1
Name: count, Length: 101, dtype: int64


In [ ]:
print(counts.value_counts())

count
1    50
2    35
3    12
4     4
Name: count, dtype: int64


In [ ]:
print(sorted.head(20))

    seismic_station gnss_station  seismic_lat  seismic_lon   gnss_lat  \
6            N.IWNH     GNET0037      38.1133     140.8441  38.317480   
4            N.YWTH     GNET0032      38.9701     140.0333  38.894600   
31           N.TAJH     GNET0176      38.5907     141.0710  38.539470   
33           N.IWNH     GNET0179      38.1133     140.8441  38.029655   
114          N.KMOH     GNET0806      37.6527     139.0676  37.776990   
115          N.KMOH     GNET0810      37.6527     139.0676  37.589737   
136          N.IWNH     GNET0919      38.1133     140.8441  38.177113   
96           N.KMOH     GNET0571      37.6527     139.0676  37.752064   
88           N.YWTH     GNET0555      38.9701     140.0333  39.015970   
84           N.TAJH     GNET0549      38.5907     141.0710  38.425050   
71           N.KMOH     GNET0238      37.6527     139.0676  37.662296   
48           N.YWTH     GNET0195      38.9701     140.0333  38.759720   
133          N.TAJH     GNET0915      38.5907     1

## sorted 기준으로 새 데이터 생성

In [ ]:
import numpy as np

# 1. 정렬된 데이터를 담을 빈 리스트 생성
aligned_seismic_data = []
aligned_seismic_stations = []
aligned_seismic_times = []

aligned_gnss_data = []
aligned_gnss_stations = []
aligned_gnss_times = []

# Filter the sorted DataFrame to include only stations present in original data
valid_seismic_stations = np.unique(seismic_station)
valid_gnss_stations = np.unique(gnss_station)

filtered_sorted = sorted[
    sorted['seismic_station'].isin(valid_seismic_stations) &
    sorted['gnss_station'].isin(valid_gnss_stations)
]

# 2. filtered_sorted 데이터프레임을 순서대로 순회하며 데이터 추출
for index, row in filtered_sorted.iterrows():
    s_station = row['seismic_station']
    g_station = row['gnss_station']

    # 원본 배열에서 단일 인덱스 찾기
    # np.where는 튜플을 반환하므로 [0][0]으로 실제 인덱스 값을 가져옴
    s_idx = np.where(seismic_station == s_station)[0][0]
    g_idx = np.where(gnss_station == g_station)[0][0]

    # Seismic 데이터 및 메타데이터 추가
    # s_idx가 단일 정수이므로 seismic_data[s_idx]는 원하는 shape의 배열을 반환
    aligned_seismic_data.append(seismic_data[s_idx])
    aligned_seismic_stations.append(seismic_station[s_idx])
    aligned_seismic_times.append(seismic_time[s_idx])

    # GNSS 데이터 및 메타데이터 추가
    aligned_gnss_data.append(gnss_data[g_idx])
    aligned_gnss_stations.append(gnss_station[g_idx])
    aligned_gnss_times.append(gnss_time[g_idx])

# 3. 리스트를 numpy 배열로 변환
new_seismic_data = np.array(aligned_seismic_data)
new_seismic_stations = np.array(aligned_seismic_stations)
new_seismic_times = np.array(aligned_seismic_times)

new_gnss_data = np.array(aligned_gnss_data)
new_gnss_stations = np.array(aligned_gnss_stations)
new_gnss_times = np.array(aligned_gnss_times)

# 4. 저장할 경로 지정
save_dir = "/content/drive/MyDrive/TriAI/그로쓰/Data/processed/multimodal/추가 실험/tohoku/"

# 5. np.savez를 사용하여 파일 저장
print("파일 저장을 시작합니다...")

np.savez(save_dir + 'new_seismic_360_180.npz',
         data=new_seismic_data,
         station_names=new_seismic_stations,
         start_times=new_seismic_times)

np.savez(save_dir + 'new_gnss_360_180.npz',
         data=new_gnss_data,
         station_names=new_gnss_stations,
         start_times=new_gnss_times)

print(f"저장 완료: {save_dir}")

파일 저장을 시작합니다...
저장 완료: /content/drive/MyDrive/TriAI/그로쓰/Data/processed/multimodal/추가 실험/tohoku/


시간 순서대로인지 확인

In [ ]:
# 검증할 인덱스 지정 (정렬된 데이터 중 첫 번째 행 확인)
test_index = 5
s_station_test = sorted.iloc[test_index]['seismic_station']
g_station_test = sorted.iloc[test_index]['gnss_station']

# 원본에서 해당 관측소의 인덱스 찾기
s_idx_test = np.where(seismic_station == s_station_test)
g_idx_test = np.where(gnss_station == g_station_test)

print(f"[정렬된 데이터 {test_index}번째 행 검증]")
print(f"Seismic Station: {s_station_test}")
print(f"GNSS Station: {g_station_test}")
print("=" * 40)

print("Seismic 데이터 8조각의 시작 시간 (원본 start_times):")
# 8개의 시간 데이터가 순서대로 나오는지 확인
for i, t in enumerate(seismic_time[s_idx_test]):
    print(f"  [{i+1}번째 조각] {t}")

print("-" * 40)

print("GNSS 데이터 8조각의 시작 시간 (원본 start_sec):")
# 8개의 시간 데이터가 순서대로 나오는지 확인
for i, t in enumerate(gnss_time[g_idx_test]):
    print(f"  [{i+1}번째 조각] {t} 초")

[정렬된 데이터 5번째 행 검증]
Seismic Station: N.KMOH
GNSS Station: GNET0810
Seismic 데이터 8조각의 시작 시간 (원본 start_times):
  [1번째 조각] ['2011-03-11T14:45:00.000000Z' '2011-03-11T14:48:00.000000Z'
 '2011-03-11T14:51:00.000000Z' '2011-03-11T14:54:00.000000Z'
 '2011-03-11T14:57:00.000000Z' '2011-03-11T15:00:00.000000Z'
 '2011-03-11T15:03:00.000000Z' '2011-03-11T15:06:00.000000Z'
 '2011-03-11T15:09:00.000000Z' '2011-03-11T15:12:00.000000Z'
 '2011-03-11T15:15:00.000000Z' '2011-03-11T15:18:00.000000Z'
 '2011-03-11T15:21:00.000000Z' '2011-03-11T15:24:00.000000Z']
----------------------------------------
GNSS 데이터 8조각의 시작 시간 (원본 start_sec):
  [1번째 조각] [   0.  180.  360.  540.  720.  900. 1080. 1260. 1440. 1620. 1800. 1980.
 2160. 2340.] 초


## PGV 저장

In [ ]:
# 저장할 경로 지정 (앞서 사용한 경로와 동일)
save_dir = "/content/drive/MyDrive/TriAI/그로쓰/Data/processed/multimodal/추가 실험/tohoku/"
csv_filename = "new_360_180_pgv.csv"

print("정렬된 PGV를 CSV로 저장 시작합니다...")

# sorted 데이터프레임을 CSV 파일로 저장 (인덱스 번호는 제외하고 저장)
filtered_sorted.to_csv(save_dir + csv_filename, index=False)

print("저장 완료")
print(f"저장된 파일: {save_dir + csv_filename}")

정렬된 PGV를 CSV로 저장 시작합니다...
저장 완료
저장된 파일: /content/drive/MyDrive/TriAI/그로쓰/Data/processed/multimodal/추가 실험/tohoku/new_360_180_pgv.csv
